# datasets

> Converters that turn common datasets into typed-decision records, multi-label sets among them, SetFit's label-only templates, splits that hold labels out for zero-shot tests, and a registry of datasets that already fit.

In [ ]:
#| default_exp datasets

In [ ]:
#| export
import json, math, random
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
from datasets import Dataset, load_dataset
from fastcore.basics import patch

from sysone.core import *
from sysone.data import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

Every application reads one record format, the typed-decisions case. Converters reshape other datasets into it once, each a short function over rows: classification sets become one choice question, pairs become a noul, ratings become a score, multi-label sets become a noul per label, and label names alone become a synthetic training set. The results feed `TypedDecisions.from_records`, or its `from_registry` shortcut, and the reshaping functions and transforms of [data](02_data.ipynb#one-label-several-questions) ask any of them in more shapes.

```{mermaid}
flowchart LR
    C["(text, label)<br/>classification"] -->|from_classification| CH["a choice:<br/>the label names as options"]
    P["(query, passage)<br/>pairs"] -->|from_pairs| NO["a noul:<br/>is the pair positive?"]
    R["ordinal ratings"] -->|from_ratings| SC["a score:<br/>the levels in order"]
    L["(text, labels)<br/>multi-label"] -->|from_multilabel| ML["a noul per label,<br/>and choices of one present label"]
    N["label names alone"] -->|from_labels| SY["choices on SetFit's<br/>templated sentences"]
    M["mojev-mix rows"] -->|from_mojev| MX["a choice, score or noul,<br/>by the option family"]
    CH & NO & SC & ML & SY & MX --> TD["TypedDecisions.from_records"]
```

## Classification

A `(text, label)` dataset becomes one choice question per row, with the label names as options, or a `{label: description}` dict when descriptions help, and a one-hot gold. The label names are read off a `ClassLabel` column when not given. This is how the evaluation suites are built.

In [ ]:
#| export
def _class_names(ds, label):
    "A ClassLabel column's names, or None"
    feat = ds.features.get(label) if hasattr(ds, "features") else None
    return list(feat.names) if feat is not None and hasattr(feat, "names") else None

def _label_names(ds, label):
    "A ClassLabel's names, else the distinct values of the column, in numeric order when they are numbers"
    names = _class_names(ds, label)
    if names is not None: return names
    vals = set(ds.unique(label) if hasattr(ds, "unique") else (r[label] for r in ds))
    numeric = all(isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool) for v in vals)
    return [str(v) for v in sorted(vals)] if numeric else sorted(str(v) for v in vals)

def _rows(ds, n=None):
    for i, r in enumerate(ds):
        if n is not None and i >= n: break
        yield r

def from_classification(ds, text="text", label:str="label", labels=None, instructions:str="Which label fits this text?",
                        qid:str="label", n:int|None=None, prefix:str="cls") -> list:
    "Choice records from a classification dataset; `text` is a column or a function of the row"
    by_index = labels is not None or _class_names(ds, label) is not None   # integer labels index these names
    names = labels if labels is not None else _label_names(ds, label)
    keys = list(names)
    criteria = names if isinstance(names, dict) else keys
    out = []
    for i, r in enumerate(_rows(ds, n)):
        y = r[label]
        is_index = by_index and isinstance(y, (int, np.integer)) and not isinstance(y, bool) and not isinstance(keys[0], (int, np.integer))
        y = keys[int(y)] if is_index else str(y)       # a class index into the names, or the label itself
        state = text(r) if callable(text) else r[text]
        out.append({"id": f"{prefix}_{i:06d}", "state": state,
                    "questions": {qid: {"type": "choice", "instructions": instructions, "criteria": criteria}},
                    "gold": {qid: {"label": y}}})
    return out

In [ ]:
toy = Dataset.from_dict({"text": ["the match ended 2-1", "shares fell 3%"], "label": [0, 1]})
from datasets import ClassLabel
toy = toy.cast_column("label", ClassLabel(names=["Sports", "Business"]))
recs = from_classification(toy, instructions="What is this news about?")
recs[1]

{'id': 'cls_000001',
 'state': 'shares fell 3%',
 'questions': {'label': {'type': 'choice',
   'instructions': 'What is this news about?',
   'criteria': ['Sports', 'Business']}},
 'gold': {'label': {'label': 'Business'}}}

In [ ]:
test_eq(recs[1]["gold"]["label"]["label"], "Business")
q = Question.from_dict(recs[0]["questions"]["label"])
test_eq(q.target(recs[0]["gold"]["label"]), ([1.0, 0.0], 0))
d = from_classification(toy, labels={"Sports": "games and athletes", "Business": "markets and companies"})
test_eq(Question.from_dict(d[0]["questions"]["label"]).options, ["Sports: games and athletes", "Business: markets and companies"])

A plain integer column (no `ClassLabel`, no names) is its own label set, in numeric order, and each row keeps its own value:

In [ ]:
plain = Dataset.from_dict({"text": [f"t{i}" for i in range(12)], "label": list(range(12))})
pr = from_classification(plain)
test_eq([r["gold"]["label"]["label"] for r in pr[:3]], ["0", "1", "2"])
test_eq(Question.from_dict(pr[0]["questions"]["label"]).keys[:3], ["0", "1", "2"])
test_eq(from_classification(Dataset.from_dict({"text": ["a"], "label": [3]}))[0]["gold"]["label"]["label"], "3")

## Pairs

A `(query, passage)` or `(premise, hypothesis)` pair becomes a noul: the two texts are the state, the instruction is the statement, and the gold is whether the pair's label is the positive one. This is the rerank and NLI shape.

In [ ]:
#| export
def from_pairs(ds, first:str="premise", second:str="hypothesis", label:str="label", positive=0,
               instructions:str="The second text follows from the first.", names=("Premise", "Hypothesis"),
               qid:str="holds", n:int|None=None, prefix:str="pair") -> list:
    "Noul records from a pair dataset: true when the row's label is `positive`"
    out = []
    for i, r in enumerate(_rows(ds, n)):
        state = f"{names[0]}: {r[first]}\n{names[1]}: {r[second]}"
        out.append({"id": f"{prefix}_{i:06d}", "state": state, "questions": {qid: {"type": "noul", "instructions": instructions}},
                    "gold": {qid: {"label": "true" if r[label] == positive else "false"}}})
    return out

In [ ]:
nli = Dataset.from_dict({"premise": ["A dog runs."], "hypothesis": ["An animal moves."], "label": [0]})
test_eq(from_pairs(nli)[0]["gold"]["holds"]["label"], "true")
from_pairs(nli)[0]

{'id': 'pair_000000',
 'state': 'Premise: A dog runs.\nHypothesis: An animal moves.',
 'questions': {'holds': {'type': 'noul',
   'instructions': 'The second text follows from the first.'}},
 'gold': {'holds': {'label': 'true'}}}

## Ratings

An ordinal rating becomes a score question with the levels as criteria, index 0 first; the gold is the level's index.

In [ ]:
#| export
def from_ratings(ds, text="text", rating:str="rating", levels:list|None=None, instructions:str="How would you rate this?",
                 qid:str="rating", offset:int|None=None, n:int|None=None, prefix:str="rating") -> list:
    "Score records from a ratings dataset; `offset` is the rating of level 0 (with `levels`, 1 for 1-based scales, else 0)"
    vals = [int(round(float(r[rating]))) for r in _rows(ds, n)]
    if levels is None:
        lo = min(vals) if offset is None else offset
        levels = [f"{v}" for v in range(lo, max(vals) + 1)]
    else: lo = offset if offset is not None else (1 if min(vals) >= 1 and max(vals) <= len(levels) else 0)
    if min(vals) - lo < 0 or max(vals) - lo >= len(levels):
        raise ValueError(f"ratings {min(vals)}..{max(vals)} do not fit {len(levels)} levels from {lo}; pass offset=")
    out = []
    for i, (r, v) in enumerate(zip(_rows(ds, n), vals)):
        state = text(r) if callable(text) else r[text]
        out.append({"id": f"{prefix}_{i:06d}", "state": state,
                    "questions": {qid: {"type": "score", "instructions": instructions, "criteria": levels}},
                    "gold": {qid: {"label": str(v - lo)}}})
    return out

In [ ]:
rv = Dataset.from_dict({"text": ["awful", "fine", "superb"], "stars": [1, 3, 5]})
stars = ["terrible", "poor", "ok", "good", "excellent"]
rr = from_ratings(rv, rating="stars", levels=stars)
test_eq([r["gold"]["rating"]["label"] for r in rr], ["0", "2", "4"])
test_eq(Question.from_dict(rr[0]["questions"]["rating"]).k, 5)
top = from_ratings(Dataset.from_dict({"text": ["ok", "great"], "stars": [3.0, 5.0]}), rating="stars", levels=stars)
test_eq([r["gold"]["rating"]["label"] for r in top], ["2", "4"])      # a 1-based scale, even when only 3..5 occur
test_fail(lambda: from_ratings(rv, rating="stars", levels=stars[:3]), contains="do not fit")

## Multi-label sets

A multi-label set gives each text any number of labels at once (the aspects a review talks about, the genres of a show). A choice can't hold that, since it has one answer, but a noul per label can: the statement that the label applies, true for the labels present. This is the binary-relevance reading of multi-label data. `labels` is a column holding each row's labels (names, or indices into a `Sequence(ClassLabel)`'s names), or a list of 0/1 columns, one per label.

Every label gives a noul by default; with `negatives=m`, a row keeps its present labels and m absent ones, drawn once. `choices=c` adds c single-answer choices, each a present label among absent ones (k options in all), a shape that does have exactly one right option. The share of true statements is the share of labels present, which can be far from a half; `question_stats` in [data](02_data.ipynb#drawn-every-epoch) counts it.

In [ ]:
#| export
def _multi_names(ds, labels):
    "The names of a multi-label column's `Sequence(ClassLabel)`, if it has them"
    feat = ds.features.get(labels) if hasattr(ds, "features") and isinstance(labels, str) else None
    inner = getattr(feat, "feature", None)
    return list(inner.names) if inner is not None and hasattr(inner, "names") else None

def from_multilabel(ds, text="text", labels="labels", names:list|None=None, statement="This text is about {label}.",
                    qid:str="label", negatives:int|None=None, choices:int=0, k:int=4,
                    instructions:str="Which of these is this text about?", n:int|None=None, prefix:str="ml", seed:int=0) -> list:
    "Records from a multi-label set: a noul per label (or per present label and `negatives` absent ones), and `choices` choices of one present label among absent ones"
    rows, rng = list(_rows(ds, n)), random.Random(seed)
    cols = None if isinstance(labels, str) else list(labels)            # one 0/1 column per label
    names = names or cols or _multi_names(ds, labels) or sorted({v for r in rows for v in r[labels]}, key=str)
    def present(r):
        if cols is not None: return [c for c in cols if r[c]]
        index = not isinstance(names[0], (int, np.integer))
        return [names[int(v)] if index and isinstance(v, (int, np.integer)) and not isinstance(v, bool) else v for v in r[labels]]
    say = (lambda x: statement[x]) if isinstance(statement, dict) else (lambda x: statement.format(label=x))
    out = []
    for i, r in enumerate(rows):
        on = present(r); off = [x for x in names if x not in on]
        keep = set(names) if negatives is None else set(on) | set(rng.sample(off, min(negatives, len(off))))
        qs = {f"{qid}:{x}": {"type": "noul", "instructions": say(x)} for x in names if x in keep}
        gold = {f"{qid}:{x}": {"label": "true" if x in on else "false"} for x in names if x in keep}
        for j in range(choices if on and off else 0):
            opts = [on[j % len(on)]] + rng.sample(off, min(k - 1, len(off)))
            gold[f"{qid}:choice{j}"] = {"label": opts[0]}
            rng.shuffle(opts)
            qs[f"{qid}:choice{j}"] = {"type": "choice", "instructions": instructions, "criteria": opts}
        out.append({"id": f"{prefix}_{i:06d}", "state": text(r) if callable(text) else r[text], "questions": qs, "gold": gold})
    return out

In [ ]:
from datasets import ClassLabel, Sequence
ml = Dataset.from_dict({"text": ["great pasta, rude waiter", "cheap and spotless"], "aspects": [[0, 1], [2, 3]]})
ml = ml.cast_column("aspects", Sequence(ClassLabel(names=["food", "service", "price", "cleanliness"])))
mr = from_multilabel(ml, labels="aspects", statement="The review talks about the {label}.", qid="aspect", choices=1)
mr[0]

{'id': 'ml_000000',
 'state': 'great pasta, rude waiter',
 'questions': {'aspect:food': {'type': 'noul',
   'instructions': 'The review talks about the food.'},
  'aspect:service': {'type': 'noul',
   'instructions': 'The review talks about the service.'},
  'aspect:price': {'type': 'noul',
   'instructions': 'The review talks about the price.'},
  'aspect:cleanliness': {'type': 'noul',
   'instructions': 'The review talks about the cleanliness.'},
  'aspect:choice0': {'type': 'choice',
   'instructions': 'Which of these is this text about?',
   'criteria': ['price', 'cleanliness', 'food']}},
 'gold': {'aspect:food': {'label': 'true'},
  'aspect:service': {'label': 'true'},
  'aspect:price': {'label': 'false'},
  'aspect:cleanliness': {'label': 'false'},
  'aspect:choice0': {'label': 'food'}}}

In [ ]:
test_eq({k: g["label"] for k, g in mr[0]["gold"].items() if "choice" not in k},
        {"aspect:food": "true", "aspect:service": "true", "aspect:price": "false", "aspect:cleanliness": "false"})
c = Question.from_dict(mr[0]["questions"]["aspect:choice0"])
test_eq((c.k, mr[0]["gold"]["aspect:choice0"]["label"]), (3, "food"))          # food among the two absent labels
test_eq(sum(k in ("food", "service") for k in c.keys), 1)                     # one present label among absent ones
few = from_multilabel(ml, labels="aspects", negatives=1)
test_eq([len(r["questions"]) for r in few], [3, 3])                           # the two present labels and one absent
flags = Dataset.from_dict({"text": ["you idiot", "nice point"], "toxic": [1, 0], "insult": [1, 0], "threat": [0, 0]})
fr = from_multilabel(flags, labels=["toxic", "insult", "threat"], statement={"toxic": "This comment is toxic.", "insult": "This comment insults someone.", "threat": "This comment threatens someone."})
test_eq([[g["label"] for g in r["gold"].values()] for r in fr], [["true", "true", "false"], ["false", "false", "false"]])
test_eq(fr[1]["questions"]["label:insult"]["instructions"], "This comment insults someone.")
named = from_multilabel([{"text": "a", "tags": ["x", "z"]}, {"text": "b", "tags": ["y"]}], labels="tags")
test_eq([sorted(k for k, g in r["gold"].items() if g["label"] == "true") for r in named], [["label:x", "label:z"], ["label:y"]])

## Label names alone

Before any data exists, SetFit's zero-shot trick bootstraps a head from the label names: `get_templated_dataset` writes `sample_size` sentences per label from a template ("This sentence is {}"), and `from_labels` wraps them as choice records. SetFit's own function is used when `setfit` is installed; otherwise an equivalent written from its documented behaviour, since the function is a small pure `datasets` recipe and SetFit's trainer is not needed.

In [ ]:
#| export
def _templated(candidate_labels, template="This sentence is {}", sample_size=2, text_column="text", label_column="label"):
    "SetFit's `get_templated_dataset` for candidate labels: `sample_size` rows per label, in label order"
    rows = [{text_column: template.format(name), label_column: i} for i, name in enumerate(candidate_labels) for _ in range(sample_size)]
    return Dataset.from_list(rows)

def get_templated_dataset(candidate_labels, template:str="This sentence is {}", sample_size:int=2):
    "Synthetic (text, label) rows from label names: SetFit's function when installed, an equivalent otherwise"
    try:
        from setfit import get_templated_dataset as setfit_templated
        return setfit_templated(candidate_labels=list(candidate_labels), template=template, sample_size=sample_size)
    except ImportError: return _templated(list(candidate_labels), template, sample_size)

def from_labels(labels, template:str="This sentence is {}", sample_size:int=2, instructions:str="Which label fits this text?",
                qid:str="label") -> list:
    "Choice records synthesised from label names (or a {label: description} dict)"
    ds = get_templated_dataset(list(labels), template, sample_size)
    return from_classification(ds, labels=labels, instructions=instructions, qid=qid, prefix="tmpl")

In [ ]:
fl = from_labels(["billing", "technical", "sales"], template="This request is about {}.")
test_eq(len(fl), 6)
test_eq((fl[0]["state"], fl[0]["gold"]["label"]["label"]), ("This request is about billing.", "billing"))
test_eq(fl[5]["gold"]["label"]["label"], "sales")

## mojev-mix

[MoLeMo-Lab/mojev-mix](https://huggingface.co/datasets/MoLeMo-Lab/mojev-mix) (MIT) holds 205,084 training rows from 18 generators, games, navigation and agent traces, each a `context` ending in a JSON state, a list of `options`, integer `preference` grades (higher is better, the correct option usually 3) and the correct option in `labels`. Rows become records: the text before `"\n\nState: "` is the instruction, the JSON after it the state; ordinal families become score questions (their options are listed in level order), a `["no", "yes"]` option pair becomes a noul, everything else a choice. mojev itself trains with a Plackett–Luce loss over the grades, not with soft targets; the soft target here, a softmax of grade / τ, is sysone's choice, so that the grades reach RLCD as a distribution.

In [ ]:
#| export
def grade_distribution(grades, tau:float=1.0) -> list:
    "Soft targets from preference grades: softmax(grade / tau)"
    g = np.asarray(grades, dtype=np.float64) / tau
    p = np.exp(g - g.max())
    return (p / p.sum()).tolist()

def mojev_record(r:dict, i:int=0, tau:float=1.0) -> dict:
    "A typed-decision record from one mojev-mix row"
    ctx = r["context"]
    ins, _, st = ctx.partition("\n\nState: ")
    try: state = json.loads(st) if st else ctx
    except json.JSONDecodeError: state = st
    opts, grades = r["options"]["answer"], r["preference"]["answer"]
    p = grade_distribution(grades, tau)
    if [o.lower() for o in opts] == ["no", "yes"]:
        q = {"type": "noul", "instructions": ins}
        gold = {"probabilities": {"false": p[0], "true": p[1]}, "label": "true" if r["labels"]["answer"].lower() == "yes" else "false"}
    elif r.get("grading") == "ordinal":
        q = {"type": "score", "instructions": ins, "criteria": list(opts)}
        gold = {"probabilities": {str(j): v for j, v in enumerate(p)}, "label": str(opts.index(r["labels"]["answer"]))}
    else:
        q = {"type": "choice", "instructions": ins, "criteria": list(opts)}
        gold = {"probabilities": dict(zip(opts, p)), "label": r["labels"]["answer"]}
    return {"id": f"mojev_{i:07d}", "source": r.get("source"), "state": state, "questions": {"answer": q}, "gold": {"answer": gold}}

def from_mojev(ds, n:int|None=None, tau:float=1.0) -> list:
    "Typed-decision records from mojev-mix rows"
    return [mojev_record(r, i, tau) for i, r in enumerate(_rows(ds, n))]

In [ ]:
row = {"context": "Choose a direction to keep the snake alive.\n\nState: {\"game\": \"snake\", \"direction\": \"up\"}",
       "options": {"answer": ["up", "right", "left"]}, "preference": {"answer": [3, 0, 0]}, "labels": {"answer": "up"},
       "source": "snake-v1", "grading": "flat"}
mr = mojev_record(row)
test_eq((mr["state"], mr["questions"]["answer"]["type"]), ({"game": "snake", "direction": "up"}, "choice"))
test_close(mr["gold"]["answer"]["probabilities"]["up"], np.exp(3) / (np.exp(3) + 2))
ordinal = dict(row, options={"answer": ["Black, 0", "Dark, 0.25", "Middle, 0.5", "Light, 0.75", "White, 1"]},
               preference={"answer": [0, 0, 1, 2, 3]}, labels={"answer": "White, 1"}, grading="ordinal")
test_eq((mojev_record(ordinal)["questions"]["answer"]["type"], mojev_record(ordinal)["gold"]["answer"]["label"]), ("score", "4"))
yn = dict(row, options={"answer": ["no", "yes"]}, preference={"answer": [0, 3]}, labels={"answer": "yes"}, grading="binary")
test_eq(Question.from_dict(mojev_record(yn)["questions"]["answer"]).target(mojev_record(yn)["gold"]["answer"])[1], 1)

## Evaluation suites

The classification sets Laya reports on, each as one choice question with the label names as options, so any model can be scored zero-shot, without task-specific training, and compared with published Laya and Jev figures. The ids are ones `datasets` 5 loads (the old un-namespaced ids and the script-based Banking77 and MASSIVE repositories no longer load); underscores in intent names become spaces, so options read as words.

In [ ]:
#| export
def _human(s): return str(s).replace("_", " ").strip()

SUITES = {
    "ag_news":   dict(path="fancyzhx/ag_news", split="test", text="text", label="label",
                      instructions="What is this news article about?"),
    "emotion":   dict(path="dair-ai/emotion", config="split", split="test", text="text", label="label",
                      instructions="Which emotion does this text express?"),
    "banking77": dict(path="legacy-datasets/banking77", split="test", text="text", label="label", human=True,
                      instructions="What does the customer want?"),
    "massive":   dict(path="mteb/amazon_massive_intent", config="en", split="test", text="text", label="label", human=True,
                      instructions="What is the intent of this request?"),
    "xnli":      dict(path="facebook/xnli", config="en", split="test", label="label",
                      text=lambda r: f"Premise: {r['premise']}\nHypothesis: {r['hypothesis']}",
                      instructions="How does the hypothesis relate to the premise?"),
}

def suite_records(name:str, n:int|None=None, split:str|None=None, seed:int=0) -> list:
    "Choice records of an evaluation suite (a sample of `n` rows when given)"
    s = SUITES[name]
    ds = load_dataset(s["path"], s.get("config"), split=split or s["split"])
    names = _label_names(ds, s["label"])          # from the whole split, before any sampling
    if n is not None and n < len(ds): ds = ds.shuffle(seed=seed).select(range(n))
    recs = from_classification(ds, s["text"], s["label"], labels=names, instructions=s["instructions"], prefix=name)
    if s.get("human"):
        for r in recs:
            r["questions"]["label"]["criteria"] = [_human(x) for x in names]
            r["gold"]["label"]["label"] = _human(r["gold"]["label"]["label"])
    return recs

In [ ]:
#| eval: false
suite_records("banking77", n=3)[0]

## Unseen labels

A zero-shot claim needs labels the model never saw. Holding out cases is not enough for that: every label still turns up in training, as an answer or as an option. `split_labels` splits records by their gold for one question instead. Records whose gold is one of the `held` labels go to the test side, as they are. The others go to the training side, with the held labels taken out of that question's options, so the model meets them nowhere.

In [ ]:
#| export
def split_labels(records, qid:str, held) -> tuple:
    "Training records, with the `held` labels taken out of question `qid`'s options, and test records, those whose gold for `qid` is a held label"
    held, seen, unseen = {str(h) for h in held}, [], []
    for r in records:
        r = load_record(r, parse_state=False)
        d = r["questions"].get(qid)
        if d is None: seen.append(r); continue
        q = Question.from_dict(d, name=qid)
        t, label = q.target((r.get("gold") or {}).get(qid))
        if t is not None and str(q.keys[label]) in held: unseen.append(r); continue
        keys = [k for k in q.keys if str(k) not in held]
        crit = keys if isinstance(d["criteria"], list) else {k: q.criteria[k] for k in keys}
        seen.append({**r, "questions": {**r["questions"], qid: {**d, "criteria": crit}}})
    return seen, unseen

In [ ]:
asks = [("wake me up at seven", "alarm_set"), ("play some jazz", "play_music"), ("will it rain tomorrow", "weather_query")]
intents = [{"id": f"i{i}", "state": s, "gold": {"intent": g},
            "questions": {"intent": {"type": "choice", "instructions": "What does the user want?",
                                     "criteria": ["alarm_set", "alarm_query", "play_music", "weather_query"]}}} for i, (s, g) in enumerate(asks)]
seen, unseen = split_labels(intents, "intent", ["weather_query", "alarm_query"])
test_eq(([r["id"] for r in seen], [r["id"] for r in unseen]), (["i0", "i1"], ["i2"]))
test_eq(seen[0]["questions"]["intent"]["criteria"], ["alarm_set", "play_music"])
test_eq(len(unseen[0]["questions"]["intent"]["criteria"]), 4)                 # the test asks among every label

## Registry

`REGISTRY` names the datasets that already fit, with where they come from, their size, what they ask, their license and their role; `TypedDecisions.from_registry(name)` loads one. Laya-browser's own training data was never published and is not here; its recipe is the browser converter in the screens notebook.

In [ ]:
#| export
@dataclass
class DatasetInfo:
    "A dataset sysone can load as typed decisions"
    name:str; source:str; size:str; modality:str; primitives:str; license:str; role:str
    load:object = field(default=None, repr=False)

REGISTRY = {}

def register_dataset(info:DatasetInfo):
    "Add a dataset to the registry"
    REGISTRY[info.name] = info
    return info

def _typed_decisions(**kw): return TypedDecisions.from_hub("LocalLLaMA/typed-decisions", **({"valid": "test"} | kw))

def _mojev(n:int|None=None, valid=None, calib=0.1, tau:float=1.0, **kw):
    ds = load_dataset("MoLeMo-Lab/mojev-mix", split="train")
    v = load_dataset("MoLeMo-Lab/mojev-mix", split="validation") if valid == "validation" else None
    return TypedDecisions.from_records(from_mojev(ds, n, tau), valid=from_mojev(v, n, tau) if v is not None else valid, calib=calib, **kw)

def _suite(name):
    def load(n:int|None=None, calib=0, **kw): return TypedDecisions.from_records(suite_records(name, n), calib=calib, **kw)
    return load

for info in [
    DatasetInfo("typed_decisions", "LocalLLaMA/typed-decisions", "1,200 train, 400 test cases, 5 questions each", "text",
                "choice, score, noul", "Apache-2.0", "Laya's fine-tune set and the reproduction target", _typed_decisions),
    DatasetInfo("mojev_mix", "MoLeMo-Lab/mojev-mix", "205,084 train, 82,423 validation, 80,703 test, 139,764 ood", "text",
                "choice with graded preferences", "MIT", "the largest ready corpus; 18 synthetic generators", _mojev),
    *[DatasetInfo(k, v["path"], "test split", "text", "choice", "per dataset", "zero-shot comparison with Laya and Jev", _suite(k))
      for k, v in SUITES.items()],
]: register_dataset(info)

@patch(cls_method=True)
def from_registry(cls:TypedDecisions, name:str, **kw):
    "A registered dataset as `TypedDecisions`"
    if name not in REGISTRY:
        import sysone.datasets  # noqa: F401  (converters registered by other notebooks)
        if name not in REGISTRY: raise ValueError(f"unknown dataset {name!r}; registered: {sorted(REGISTRY)}")
    out = REGISTRY[name].load(**kw)
    out.source = {"registry": name} | (out.source or {})
    return out

In [ ]:
sorted(REGISTRY)

['ag_news',
 'banking77',
 'emotion',
 'massive',
 'mojev_mix',
 'typed_decisions',
 'xnli']

In [ ]:
test_eq(REGISTRY["typed_decisions"].license, "Apache-2.0")
test_fail(lambda: TypedDecisions.from_registry("imagenet"), contains="unknown dataset")

In [ ]:
#| eval: false
TypedDecisions.from_registry("typed_decisions")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()